# Shap evaluation

Kendra Wyant  
April 29, 2026

### Set Up Environment

In [ ]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(source("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true"))
suppressPackageStartupMessages(library(tidyposterior))

path_models <- format_path(str_c("risk2/models/combined"))
path_shared <- format_path(str_c("risk2/data_processed/shared"))

## Read in data

In [ ]:
shaps <- read_rds(here::here(path_models, 
                             "outer_shapsgrp_6_x_5_1_x_5_day_v10_nested_full.rds"))

labels <- read_csv(here::here(path_shared, "lapse_labels_24h_day_gps.csv"),
                   show_col_types = FALSE)

dem <- read_csv(here::here(path_shared, "features_fairness.csv"),
                 show_col_types = FALSE) |>
  select(-ruca1) |> 
  filter(subid %in% labels$subid)

## Match `id_obs` to fairness subgroups

In [ ]:
shap_dem <- labels |> 
  select(id_obs = label_num,
         label = lapse,
         subid) |>
  right_join(shaps, by = c("id_obs")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse"))) |> 
  left_join(dem, by = "subid") |> 
  glimpse()

Rows: 40,682,842
Columns: 12
$ id_obs       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ label        <fct> No lapse, No lapse, No lapse, No lapse, No lapse, No laps…
$ subid        <dbl> 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 1006, 100…
$ variable_grp <chr> "Abstinence confidence, Diff (EMA)", "Abstinence confiden…
$ split_num    <int> 4, 7, 13, 16, 22, 27, 4, 7, 13, 16, 22, 27, 4, 7, 13, 16,…
$ value        <dbl> -0.0692082778, -0.0345855913, -0.1322320967, -0.027555609…
$ rfvalue      <dbl> -0.070734091, -0.077360266, -0.057233856, -0.061833461, -…
$ geography    <chr> "urban/suburban", "urban/suburban", "urban/suburban", "ur…
$ race         <chr> "non-Hispanic White", "non-Hispanic White", "non-Hispanic…
$ income       <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, N…
$ gender       <chr> "male", "male", "male", "male", "male", "male", "male", "…
$ orientation  <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, N…

## Get top 10 global shaps across sample

In [ ]:
(shaps_10 <- shaps |> 
  group_by(variable_grp) |>
  summarize(mean_value = mean(abs(value)), .groups = "drop") |> 
  arrange(desc(mean_value)) |> 
  slice_head(n = 10))

# A tibble: 10 × 2
   variable_grp                      mean_value
   <chr>                                  <dbl>
 1 Past opioid use, Diff (EMA)           0.828 
 2 Urge, Raw (EMA)                       0.355 
 3 Abstinence confidence, Raw (EMA)      0.316 
 4 Stimulant use, Diff (EMA)             0.138 
 5 Pleasant event, Diff (EMA)            0.0814
 6 Abstinence motivation, Raw (EMA)      0.0765
 7 Helpful locations, Diff (gps)         0.0617
 8 Harmful locations, Diff (gps)         0.0502
 9 Urge, Diff (EMA)                      0.0470
10 Abstinence confidence, Diff (EMA)     0.0450

## Create test data frame for Past opioid use, Diff (EMA)

In [ ]:
shap_past_use_race <- shap_dem |> 
  filter(variable_grp == "Past opioid use, Diff (EMA)" & !is.na(race)) |> 
  group_by(race, split_num, variable_grp) |> 
  summarize(mean_value = mean(abs(value)), .groups = "drop") |> 
  glimpse()

Rows: 60
Columns: 4
$ race         <chr> "Hispanic and/or not white", "Hispanic and/or not white",…
$ split_num    <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17…
$ variable_grp <chr> "Past opioid use, Diff (EMA)", "Past opioid use, Diff (EM…
$ mean_value   <dbl> 0.9421857, 0.8664163, 0.9115160, 0.6958182, 1.0321347, 0.…

Rows: 30
Columns: 5
$ repeat_num                  <dbl> 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, …
$ fold_num                    <int> 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, …
$ variable_grp                <chr> "Past opioid use, Diff (EMA)", "Past opioi…
$ `Hispanic and/or not white` <dbl> 0.9421857, 0.8664163, 0.9115160, 0.6958182…
$ `non-Hispanic White`        <dbl> 0.9785541, 0.9241072, 0.8590925, 0.7900985…

#### Bayesian comparison

In [ ]:
#| output: false

# Repeated CV (id = repeat, id2 = fold within repeat)
# with a common variance:  statistic ~ model + (model | id2/id)
set.seed(101)
pp <- shap_past_use_race |>
  select(-variable_grp) |>
  rename(id = fold_num,
         id2 = repeat_num) |> 
  perf_mod(formula = statistic ~ model + (1 | id2/id),
        # transform = tidyposterior::logit_trans,  # for skewed & bounded AUC
         iter = 8000, chains = 4, adapt_delta = .99, # increased iteration from 2000 to fix divergence issues
         family = gaussian, 
)  


SAMPLING FOR MODEL 'continuous' NOW (CHAIN 1).
Chain 1: 
Chain 1: Gradient evaluation took 8.6e-05 seconds
Chain 1: 1000 transitions using 10 leapfrog steps per transition would take 0.86 seconds.
Chain 1: Adjust your expectations accordingly!
Chain 1: 
Chain 1: 
Chain 1: Iteration:    1 / 8000 [  0%]  (Warmup)
Chain 1: Iteration:  800 / 8000 [ 10%]  (Warmup)
Chain 1: Iteration: 1600 / 8000 [ 20%]  (Warmup)
Chain 1: Iteration: 2400 / 8000 [ 30%]  (Warmup)
Chain 1: Iteration: 3200 / 8000 [ 40%]  (Warmup)
Chain 1: Iteration: 4000 / 8000 [ 50%]  (Warmup)
Chain 1: Iteration: 4001 / 8000 [ 50%]  (Sampling)
Chain 1: Iteration: 4800 / 8000 [ 60%]  (Sampling)
Chain 1: Iteration: 5600 / 8000 [ 70%]  (Sampling)
Chain 1: Iteration: 6400 / 8000 [ 80%]  (Sampling)
Chain 1: Iteration: 7200 / 8000 [ 90%]  (Sampling)
Chain 1: Iteration: 8000 / 8000 [100%]  (Sampling)
Chain 1: 
Chain 1:  Elapsed Time: 7.195 seconds (Warm-up)
Chain 1:                10.053 seconds (Sampling)
Chain 1:                17.

In [ ]:
pp_tidy <- pp |> 
  tidy(seed = 123) 

q = c(.025, .5, .975)
pp_perf_tibble <- pp_tidy |> 
  group_by(model) |> 
  summarize(pp_median = quantile(posterior, probs = q[2]),
            pp_lower = quantile(posterior, probs = q[1]), 
            pp_upper = quantile(posterior, probs = q[3])) |> 
  mutate(model = factor(model, levels = c("Hispanic and/or not white", "non-Hispanic White"))) |> 
  arrange(model) 

pp_perf_tibble

# A tibble: 2 × 4
  model                     pp_median pp_lower pp_upper
  <fct>                         <dbl>    <dbl>    <dbl>
1 Hispanic and/or not white     0.818    0.740    0.898
2 non-Hispanic White            0.837    0.759    0.916

### Fairness Comparison

In [ ]:
ci <- pp |>
  contrast_models(list("Hispanic and/or not white"),
                  list("non-Hispanic White")) |>
  summary(size = 0) 

ci_median <- pp |>
  contrast_models(list("Hispanic and/or not white"),
                  list("non-Hispanic White")) |>
  group_by(contrast) |>
  summarize(median = quantile(difference, .5))


ci <- ci |>
  left_join(ci_median, by = c("contrast")) 

ci

# A tibble: 1 × 10
  contrast      probability    mean   lower    upper  size pract_neg pract_equiv
  <chr>               <dbl>   <dbl>   <dbl>    <dbl> <dbl>     <dbl>       <dbl>
1 Hispanic and…      0.0432 -0.0192 -0.0375 -8.19e-4     0        NA          NA
# ℹ 2 more variables: pract_pos <dbl>, median <dbl>